In [1]:
# Cell 1: Mount Google Drive and set project paths
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/financial-news-sentiment-stock-predictor'
DATA_PATH = f'{PROJECT_PATH}/data'

print("Drive mounted successfully")
print(f"Project path: {PROJECT_PATH}")

Mounted at /content/drive
Drive mounted successfully
Project path: /content/drive/MyDrive/financial-news-sentiment-stock-predictor


In [2]:
# Cell 2: Install and import required libraries
!pip install yfinance pandas numpy -q

import yfinance as yf
import pandas as pd
import numpy as np
import os

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"yfinance version: {yf.__version__}")

Libraries imported successfully
Pandas version: 2.2.2
yfinance version: 0.2.66


In [3]:
# Cell 3: Define multi-stock parameters and fetch historical OHLCV data
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
PERIOD = '5y'
INTERVAL = '1d'

all_price_data = []

for ticker in TICKERS:
    print(f"Fetching {ticker}...")
    raw = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)

    # Flatten multi-level columns
    raw.columns = raw.columns.get_level_values(0)
    raw.columns.name = None

    # Reset index so Date becomes a column
    raw = raw.reset_index()

    # Add ticker identifier column
    raw['ticker'] = ticker

    all_price_data.append(raw)
    print(f"  {ticker}: {len(raw)} rows | "
          f"{raw['Date'].min().date()} to {raw['Date'].max().date()}")

# Combine all tickers into one DataFrame
price_data = pd.concat(all_price_data, ignore_index=True)

print(f"\nCombined dataset shape: {price_data.shape}")
print(f"\nRows per ticker:")
print(price_data['ticker'].value_counts().sort_index())

Fetching AAPL...


/tmp/ipykernel_2562/3871425279.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)


  AAPL: 1256 rows | 2021-06-14 to 2026-06-12
Fetching MSFT...


/tmp/ipykernel_2562/3871425279.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)


  MSFT: 1256 rows | 2021-06-14 to 2026-06-12
Fetching GOOGL...


...

  GOOGL: 1256 rows | 2021-06-14 to 2026-06-12
Fetching AMZN...


/tmp/ipykernel_2562/3871425279.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)


  AMZN: 1256 rows | 2021-06-14 to 2026-06-12
Fetching TSLA...


/tmp/ipykernel_2562/3871425279.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)


  TSLA: 1256 rows | 2021-06-14 to 2026-06-12

Combined dataset shape: (6280, 7)

Rows per ticker:
ticker
AAPL     1256
AMZN     1256
GOOGL    1256
MSFT     1256
TSLA     1256
Name: count, dtype: int64


In [4]:
# Cell 4: Clean and validate the combined price dataset
# Sort by ticker and date - critical for rolling window calculations later
price_data = price_data.sort_values(
    ['ticker', 'Date']).reset_index(drop=True)

# Convert Date to datetime
price_data['Date'] = pd.to_datetime(price_data['Date'])

# Check for missing values per ticker
print("Missing values per column:")
print(price_data.isnull().sum())

print("\nDuplicate rows:", price_data.duplicated(
    subset=['Date', 'ticker']).sum())

# Check each ticker's date range
print("\nDate ranges per ticker:")
for ticker in TICKERS:
    subset = price_data[price_data['ticker'] == ticker]
    print(f"  {ticker}: {subset['Date'].min().date()} "
          f"to {subset['Date'].max().date()} "
          f"| {len(subset)} rows")

# Basic price sanity check - no zero or negative prices
print("\nNegative/zero prices:")
price_cols = ['Open', 'High', 'Low', 'Close']
for col in price_cols:
    invalid = (price_data[col] <= 0).sum()
    print(f"  {col}: {invalid} invalid values")

price_data.head(10)

Missing values per column:
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
ticker    0
dtype: int64

Duplicate rows: 0

Date ranges per ticker:
  AAPL: 2021-06-14 to 2026-06-12 | 1256 rows
  MSFT: 2021-06-14 to 2026-06-12 | 1256 rows
  GOOGL: 2021-06-14 to 2026-06-12 | 1256 rows
  AMZN: 2021-06-14 to 2026-06-12 | 1256 rows
  TSLA: 2021-06-14 to 2026-06-12 | 1256 rows

Negative/zero prices:
  Open: 0 invalid values
  High: 0 invalid values
  Low: 0 invalid values
  Close: 0 invalid values


,Date,Close,High,Low,Open,Volume,ticker
0,2021-06-14,127.185646,127.244129,123.861745,124.592809,96906500,AAPL
1,2021-06-15,126.366859,127.302628,126.123171,126.659288,62746300,AAPL
2,2021-06-16,126.863976,127.585297,125.216657,127.078422,91815000,AAPL
3,2021-06-17,128.462509,129.203330,126.376541,126.522763,96721700,AAPL
4,2021-06-18,127.166115,128.189592,126.951668,127.409803,108953300,AAPL
5,2021-06-21,128.959702,129.066925,125.947721,127.010197,79663300,AAPL
6,2021-06-22,130.597260,130.694741,128.296844,128.793977,74783600,AAPL
7,2021-06-23,130.324280,130.928636,129.866145,130.392520,60214200,AAPL
8,2021-06-24,130.041672,131.240613,129.573780,131.055407,68711000,AAPL
9,2021-06-25,129.749237,130.509542,129.456808,130.090406,70783700,AAPL


In [5]:
# Cell 5: Create binary target variable per ticker
# Must compute next-day direction WITHIN each ticker separately
price_data['target'] = (
    price_data.groupby('ticker')['Close']
    .shift(-1) > price_data['Close']
).astype(int)

# Drop last row of each ticker (no next day available)
price_data = price_data.groupby('ticker').apply(
    lambda x: x.iloc[:-1]).reset_index(drop=True)

# Verify target distribution per ticker
print("Target distribution per ticker:")
for ticker in TICKERS:
    subset = price_data[price_data['ticker'] == ticker]
    up = subset['target'].sum()
    down = len(subset) - up
    print(f"  {ticker}: {len(subset)} rows | "
          f"Up={up} ({up/len(subset)*100:.1f}%) | "
          f"Down={down} ({down/len(subset)*100:.1f}%)")

print(f"\nOverall target distribution:")
print(price_data['target'].value_counts())
print(f"\nOverall class balance:")
print(price_data['target'].value_counts(normalize=True) * 100)

Target distribution per ticker:
  AAPL: 1255 rows | Up=668 (53.2%) | Down=587 (46.8%)
  MSFT: 1255 rows | Up=647 (51.6%) | Down=608 (48.4%)
  GOOGL: 1255 rows | Up=666 (53.1%) | Down=589 (46.9%)
  AMZN: 1255 rows | Up=643 (51.2%) | Down=612 (48.8%)
  TSLA: 1255 rows | Up=653 (52.0%) | Down=602 (48.0%)

Overall target distribution:
target
1    3277
0    2998
Name: count, dtype: int64

Overall class balance:
target
1    52.223108
0    47.776892
Name: proportion, dtype: float64


/tmp/ipykernel_2562/2259747049.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  price_data = price_data.groupby('ticker').apply(


In [6]:
# Cell 6: Final check and save multi-stock price data
print("Final dataset info:")
print(f"Shape: {price_data.shape}")
print(f"Columns: {price_data.columns.tolist()}")
print(f"Tickers: {price_data['ticker'].unique().tolist()}")
print(f"Date range: {price_data['Date'].min().date()} "
      f"to {price_data['Date'].max().date()}")

# Save to CSV
price_data.to_csv(f'{DATA_PATH}/price_data.csv', index=False)
print(f"\nSaved to {DATA_PATH}/price_data.csv")
print(f"Total rows saved: {len(price_data)}")

Final dataset info:
Shape: (6275, 8)
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'ticker', 'target']
Tickers: ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'TSLA']
Date range: 2021-06-14 to 2026-06-11

Saved to /content/drive/MyDrive/financial-news-sentiment-stock-predictor/data/price_data.csv
Total rows saved: 6275
